In [1]:
### Run MOFA Model on the data generated in the previous script (02)

#############################################
# Prerequisites - Load Libraries

In [2]:
source('MS1_Functions.r')

In [3]:
### Informa about start of execution

popup_function_pos('03_Run_MOFA: Execution Started')

In [4]:
source('MS0_Libraries.r')

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/R/library/"


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘tibble’ was built under R version 4.3.3”
Warning message:
“package ‘purrr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.3”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
Warning message:
“package ‘lubridate’ was built under R version 4.3.3”
── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“packa

In [5]:
source('MS2_Plot_Config.r')

Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


In [6]:
py_config() # - To check the configuration which python package will be used for MOFA

python:         /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/bin/python
libpython:      /home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/libpython3.8.so
pythonhome:     /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env:/ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env
version:        3.8.19 (default, Mar 20 2024, 19:58:24)  [GCC 11.2.0]
numpy:          /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by RETICULATE_PYTHON

###############################################
# Preqrequisites Configurations & Parameters

In [7]:
### Load the parameters that are set via the configuration files

In [8]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [9]:
head(global_configs,2)

,parameter,value
,<chr>,<chr>
1,data_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/
2,result_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/


In [10]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [11]:
data_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/"

In [12]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [13]:
result_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/"

In [14]:
## MOFA Model Configurations

In [15]:
mofa_configs = read.csv( 'configurations/03_MOFA_Configs.csv', sep = ',')
mofa_configs = mofa_configs[mofa_configs$configuration_name != '',]

In [16]:
head(mofa_configs,2)

,configuration_name,mofa_result_name,amount_of_factors,weighting_of_views,scale_views
,<chr>,<chr>,<int>,<lgl>,<lgl>
1,CGS_v3,CGS_v3_MOFA,13,FALSE,TRUE


In [17]:
### Generate the result data directory if it does not exist yet
if(!file.exists(paste0(result_path, '03_results'))){
    dir.create(file.path(paste0(result_path, '03_results')))
    }

# Load Data 

## Prepared combined data

In [18]:
### Load the data that was generated in the previous script using the name specified in the configuration file

In [19]:
input_data = list()

In [20]:
for(i in 1:nrow(mofa_configs)){
    path = paste0(result_path, '/02_results/02_Combined_Data_', mofa_configs$configuration_name[i] ,'_INTEGRATED.csv')
    
    if(file.exists(path)){
        data_long = read.csv(path)
        data_long$X = NULL
        print(path)
        print(file.info(path)$mtime)
        input_data[[i]]= data_long
        popup_function_pos('Prepared Data has been loaded')
        }
    else{
        popup_function_neg(paste0( 'Error: Data at' , result_path, '/02_results/', ' 02_Combined_Data_', mofa_configs$configuration_name[i] ,'_INTEGRATED.csv', ' could not be loaded. Make sure that all the previous steps have been executed successfully'))
     }   
        
    }

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//02_results/02_Combined_Data_CGS_v3_INTEGRATED.csv"
[1] "2024-09-23 18:39:28 CEST"


In [21]:
length(unique(input_data[[1]]$variable))

[1] 9618

In [22]:
unique(input_data[[1]]$type)

[1] "Activated.T.cells"       "B.Cells"                
 [3] "Classical.Monocytes"     "clinical"               
 [5] "Dendritic.cells"         "NK.cells"               
 [7] "Non.Classical.Monocytes" "proteomic"              
 [9] "Regulatory.T.cells"      "T.cells"

In [23]:
length(unique(input_data[[1]]$sample_id))

[1] 99

# Train MOFA Model

## Prepare data list

In [24]:
### Adjust single-cell types to correspond to cell-types

In [25]:
head(input_data[[1]],2)

,sample_id,variable,value,type,gene
,<chr>,<chr>,<dbl>,<chr>,<chr>
1,HF10,Activated.T.cells__AAK1,0.4022501,Activated.T.cells,AAK1
2,HF11,Activated.T.cells__AAK1,0.0000000,Activated.T.cells,AAK1


In [26]:
### Prepare data list for MOFA (adjust format of input data to be used as input for MOFA)

In [27]:
input_data_list = list()

In [28]:
data_list = list()

In [29]:
input_data_list = lapply(input_data, function(x){

    for(i in unique(x$type)){
        samples = unique(x$sample_id) # necessary to have all samples in all dimensions
        data = x[x$type == i, ]

        data$type = NULL
        data$cell_type = NULL

        data = data %>% dcast(variable ~ sample_id, value  = "value")
        rownames(data) = data$variable
        colnames(data) = str_replace(colnames(data), 'value\\.', '')
        data$variable = NULL

        data[setdiff( samples, names(data))] = NA  # use all samples

        data = data[,order(colnames(data))]
        data = data[,colnames(data) %in% samples]

        data_list[[i]] = as.matrix(data)
        }
    
    return(data_list)
    })

In [30]:
head(input_data_list[[1]][[1]],2)

,HF1,HF10,HF11,HF12,HF13,HF14,HF15,HF16,HF2,HF3,⋯,KS6.1,KS6.2,KS6.3,KS7.1,KS7.2,KS8.1,KS8.2,KS8.3,KS9.1,KS9.2
Activated.T.cells__AAK1,NA,0.4022501,0.000000,-0.15731068,0.5485223,-0.6420614,0.7245144,-0.6102946,-0.3740954,2.310991,⋯,-0.07841241,1.6249807,-1.1011455,1.53412054,-0.8122178,0.4887764,1.202508,NA,-1.2581616,-0.51841799
Activated.T.cells__ABLIM1,NA,0.7076435,1.624981,0.02611368,-1.3829941,-0.7415940,0.8305109,0.8305109,-0.4164465,-1.202508,⋯,-0.31863936,-0.8679592,0.4741167,0.05224518,-1.0099902,-1.1503494,-1.454408,NA,-0.9674216,0.09151507


In [31]:
#str(input_data_list)

## Create MOFA object

In [32]:
### Create a MOFA object to run the MOFA model on it

In [33]:
names(input_data_list[[1]])

[1] "Activated.T.cells"       "B.Cells"                
 [3] "Classical.Monocytes"     "clinical"               
 [5] "Dendritic.cells"         "NK.cells"               
 [7] "Non.Classical.Monocytes" "proteomic"              
 [9] "Regulatory.T.cells"      "T.cells"

In [34]:
mofa_object = lapply(input_data_list, function(x){
    MOFAobject = create_mofa(x)
    }
                     )

Creating MOFA object from a list of matrices (features as rows, sample as columns)...




In [35]:
### Plot the Data Overview showing the input used for the MOFA Model

In [36]:
# Specific Text Descriptions:
xlabel = xlab('Factors') 
ylabel = ylab('Views')

In [37]:
# Sizes of the plot
width_par = 5
height_par =5

In [38]:
options(repr.plot.width=30, repr.plot.height=10)

mofa_overview = lapply(mofa_object, function(x){
    mofa_overview = plot_data_overview(x)
    mofa_overview = mofa_overview + plot_config + theme(axis.text.y = element_text(hjust = 0, vjust = 0.5)) +
                xlabel + ylabel + theme(axis.text.x = element_blank())
    })

In [39]:
#mofa_overview[[1]]

In [40]:
# Extract data -type colors (used by the function to align and use those colors in the next plots)
type_colors = list()
for(i in 1:length(mofa_overview)){
    color_extraction =  ggplot_build(mofa_overview[[i]])
    type_colors[[i]] = unique(color_extraction$data[[1]]["fill"][,1])
    type_colors[[i]] = type_colors[[i]][!type_colors[[i]] == 'grey']
    }
    

In [41]:
type_colors

[[1]]
 [1] "#FF7F50" "#D95F02" "#377EB8" "#E6AB02" "#31A354" "#7570B3" "#E7298A"
 [8] "#66A61E" "#A6761D" "#666666"

In [42]:
figure_name = "FIG03_Overview_MOFA_Input_"

In [43]:

for(i in 1:length(mofa_overview)){
    pdf(paste0('figures/03_figures/', figure_name, mofa_configs$mofa_result[i],  '.pdf'), width =width_par, height =height_par)
    print(mofa_overview[[i]] )
    dev.off()
    }
popup_function_pos(paste0('Generated: figures/03_figures/ ', figure_name, mofa_configs$mofa_result[i],  '.pdf'))

## Set MOFA Training Options and run the Model Training

In [44]:
### Define the MOFA parameters for training and run the model training with the set parameters
### Some parameters are handed over by thhe configuration file
### Others are currently assigned fixed below but can be modified

In [45]:
model_result = list()

In [46]:
for(i in 1:length(mofa_object)){
    
    ## Set other parameters of MOFA Model
    mefisto_opts = get_default_mefisto_options(mofa_object[[i]])
    
    ## Data Options
    data_opts = get_default_data_options(mofa_object[[i]])
    data_opts$scale_views = mofa_configs$scale_views[i] # decide whether to scale the data
    data_opts$use_float32 = FALSE
    print(data_opts)
    
    ## Model Options
    model_opts = get_default_model_options(mofa_object[[i]])
    model_opts$num_factors = mofa_configs$amount_of_factors[i] # define number of factors
    model_opts$spikeslab_weights = TRUE
    # model_opts$likelihoods['neutrophil'] = 'poisson' - example to modify distribution for one specific view
    print(model_opts)
    
    ## Training Options
    train_opts  = get_default_training_options(mofa_object[[i]])
    train_opts$maxiter = 50000
    train_opts$verbose = TRUE
    train_opts$seed = 42
    train_opts$weight_views = mofa_configs$weighting_of_views[i]
    print(train_opts)
    
    ## Build and train the model
    MOFAobject = prepare_mofa(
      object = mofa_object[[i]],
      data_options = data_opts,
      model_options = model_opts,
      mefisto_options = mefisto_opts,
      training_options = train_opts #,
      #stochastic_options = stoch_options
    )
    
    model_name = paste0("03_MOFA_MODEL_", mofa_configs$mofa_result[i], '.hdf5')
    outfile = file.path( paste0(result_path, '/03_results/',  model_name) )
    print(outfile)
    MOFAobject.trained = run_mofa(MOFAobject, outfile, use_basilisk = TRUE)
    

    model_result[[i]] = MOFAobject.trained
    
    }
    

$scale_views
[1] TRUE

$scale_groups
[1] FALSE

$center_groups
[1] TRUE

$use_float32
[1] FALSE

$views
 [1] "Activated.T.cells"       "B.Cells"                
 [3] "Classical.Monocytes"     "clinical"               
 [5] "Dendritic.cells"         "NK.cells"               
 [7] "Non.Classical.Monocytes" "proteomic"              
 [9] "Regulatory.T.cells"      "T.cells"                

$groups
[1] "group1"

$likelihoods
      Activated.T.cells                 B.Cells     Classical.Monocytes 
             "gaussian"              "gaussian"              "gaussian" 
               clinical         Dendritic.cells                NK.cells 
             "gaussian"              "gaussian"              "gaussian" 
Non.Classical.Monocytes               proteomic      Regulatory.T.cells 
             "gaussian"              "gaussian"              "gaussian" 
                T.cells 
             "gaussian" 

$num_factors
[1] 13

$spikeslab_factors
[1] FALSE

$spikeslab_weights
[1] TRUE

$ard_f

Warning message in prepare_mofa(object = mofa_object[[i]], data_options = data_opts, :
“Some view(s) have less than 15 features, MOFA will have little power to to learn meaningful factors for these view(s)....”
Checking data options...

Checking training options...

Checking model options...



[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//03_results/03_MOFA_MODEL_CGS_v3_MOFA.hdf5"



Connecting to the mofapy2 package using basilisk. 
    Set 'use_basilisk' to FALSE if you prefer to manually set the python binary using 'reticulate'.



In [47]:
#str(model_result)

In [48]:
reticulate::py_config()

python:         /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/bin/python
libpython:      /home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/libpython3.8.so
pythonhome:     /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env:/ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env
version:        3.8.19 (default, Mar 20 2024, 19:58:24)  [GCC 11.2.0]
numpy:          /ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by RETICULATE_PYTHON

# Extract and prepare data for plots

In [49]:
### Extract generated data for the model to use for later downstream analysis

## Extract Variance decomposition

In [50]:
# Extract the total explained variance per view and factor

In [51]:
model_result[[1]]@cache[["variance_explained"]]$r2_total  # per view

$group1
      Activated.T.cells                 B.Cells     Classical.Monocytes 
               47.93013                34.26799                44.45263 
               clinical         Dendritic.cells                NK.cells 
               27.02723                35.08060                43.47338 
Non.Classical.Monocytes               proteomic      Regulatory.T.cells 
               28.50707                12.25573                50.56718 
                T.cells 
               58.81582

In [52]:
rowMeans(model_result[[1]]@cache$variance_explained$r2_per_factor[[1]]) # per factor

Factor1  Factor2  Factor3  Factor4  Factor5  Factor6  Factor7  Factor8 
9.931573 5.975114 4.726286 2.696757 2.063430 2.050188 1.965255 1.962400 
 Factor9 Factor10 Factor11 Factor12 Factor13 
1.846568 1.483068 1.479192 1.214973 1.154617

In [53]:
# Mean total variance explained

In [54]:
mean(model_result[[1]]@cache$variance_explained$r2_total[[1]])

[1] 38.23778

In [55]:
# Save the explained variance

In [56]:
for(i in 1:length(model_result)){
    write.csv(model_result[[i]]@cache$variance_explained$r2_per_factor[[1]], paste0(result_path, '/03_results/03_MOFA_Variance_Decomposition_',mofa_configs$mofa_result[i], '.csv'))
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_MOFA_Variance_Decomposition_',mofa_configs$mofa_result[i], '.csv'))

## Extract factor and weight data

In [57]:
#### Extract sample factors  values and save

In [58]:
for(i in 1:length(model_result)){
    factors = get_factors(model_result[[i]], factors = "all")
    factors = factors$group1
    head(factors,2)
    
    factors = as.data.frame(factors)
    factors$sample_id = rownames(factors)
    
    # Save as csv
    write.csv(factors, paste0(result_path, '/03_results/03_Factor_Data_' , mofa_configs$mofa_result[i],  '.csv'), row.names = FALSE)
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_Factor_Data_' , mofa_configs$mofa_result[i],  '.csv'))

In [59]:
### Extract weight data (feature factor weights) and save

In [60]:
for(j in 1:length(model_result)){
    weights = get_weights(model_result[[j]], views = "all", factors = "all")
    weight_data = data.frame()
    
    for (i in names(weights)){
        data = data.frame(weights[[i]])
        data$type = i
        weight_data = rbind(weight_data,data)
        }
    weight_data$variable_name = rownames(weight_data)
    
    # Save as csv
    write.csv(weight_data, paste0(result_path, '/03_results/03_Weight_Data_' ,mofa_configs$mofa_result[j], '.csv'), row.names = FALSE)
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_Weight_Data_' ,mofa_configs$mofa_result[j], '.csv'))

# Diagnostic Result Plots

In [61]:
### Make the explained variance plot to analyze the model result

## Plot explained variance overview

In [62]:
## Prepare the data format

In [63]:
# Access the first model in the list
model <- model_result[[1]]

# Access the explained variance per factor for the first factor (or view)
# Note: This assumes the structure of the model allows accessing @cache and variance_explained this way
data <- model@cache$variance_explained$r2_per_factor[[1]]

# View the extracted data
print(data)


         Activated.T.cells    B.Cells Classical.Monocytes    clinical
Factor1         15.8583813 10.6527734          8.10100652  0.78764102
Factor2          7.7028083  6.8605031          8.07536661  0.03555806
Factor3          5.9603441  2.7674801          8.27229143 12.79814650
Factor4          1.9568825  1.2714494          2.57644493  6.34315202
Factor5          0.6875394  1.3520625          0.94094206  1.40955158
Factor6          3.4423809  2.2298828          2.02865308  0.01797487
Factor7          2.2412670  1.8110151          3.55977119  1.20003817
Factor8          1.9989509  2.0077122          4.21654427  0.12744878
Factor9          2.5845595  1.2505215          0.75074471  0.90362409
Factor10         1.3366193  0.7695408          0.49890197  1.98079592
Factor11         1.7359880  2.0386695          0.07759598  0.94318617
Factor12         2.2960239  1.1922681          1.17827027  0.30127158
Factor13         0.4435464  0.3900158          4.93296141  0.04116978
         Dendritic.c

In [64]:
explained_variance = lapply(model_result, function(x) {
    data = x@cache$variance_explained$r2_per_factor[[1]]
    data = melt(data)
    print(names(x@cache[["variance_explained"]]$r2_total$group1))
    print(x@cache[["variance_explained"]]$r2_total$group1)
    total_variance = data.frame( view = names(x@cache[["variance_explained"]]$r2_total$group1),
                             total_variance = x@cache[["variance_explained"]]$r2_total$group1)
    data = merge(data, total_variance, by.x = 'Var2', by.y = 'view')
    data$Var2 = as.character(data$Var2)
    data$Var2 = factor(data$Var2, levels = sort(unique(data$Var2)))
    data = data[order(data$Var2),]
    }
                            )

 [1] "Activated.T.cells"       "B.Cells"                
 [3] "Classical.Monocytes"     "clinical"               
 [5] "Dendritic.cells"         "NK.cells"               
 [7] "Non.Classical.Monocytes" "proteomic"              
 [9] "Regulatory.T.cells"      "T.cells"                
      Activated.T.cells                 B.Cells     Classical.Monocytes 
               47.93013                34.26799                44.45263 
               clinical         Dendritic.cells                NK.cells 
               27.02723                35.08060                43.47338 
Non.Classical.Monocytes               proteomic      Regulatory.T.cells 
               28.50707                12.25573                50.56718 
                T.cells 
               58.81582 


In [65]:
#### Plot complete explained variance (Heatmap)

In [66]:
var_decomp = lapply(explained_variance, function(x){
    ggplot() + 
        scale_fill_gradient(low="white", high="black") + 
        xlabel + 
        ylabel +
        plot_config + theme(axis.text.x = element_text(angle = 90), legend.position = 'right')+ 
        geom_tile(data = x, mapping = aes(Var1,  Var2, fill= value))
    })

In [67]:
### Combine the plot with total variance barplot per dimension

In [68]:
# Specific Text Descriptions:
xlabel = xlab('Views') 
ylabel = ylab('Explained Variance (%)')

In [69]:
comp_variance = lapply(explained_variance, function(x){
    data = x
    plot_complete = unique(data[,c('Var2', 'total_variance')])
    comp_variance = ggplot(plot_complete, aes(x=Var2, y = total_variance, fill = Var2)) + 
                geom_bar(stat="identity") + coord_flip() + 
                xlabel + 
                ylabel +
                plot_config + scale_fill_manual(values = unlist(type_colors))  ## currently uses same coloring as MOFA oveview
    })

In [70]:
#comp_variance[[1]]

In [71]:
### Combine both visualization

In [72]:
figure_name = "FIG03_Overview_Variance_Decomposition"

In [73]:
# Sizes of the plot
width_par = 8.07
height_par = 4  # 2.8

In [74]:
for(i in 1:length(explained_variance)){
    legend = get_legend(var_decomp[[i]])
    
    combination1 = ggarrange(var_decomp[[i]] + theme(legend.position = 'none'),
                     comp_variance[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank(), legend.position = 'none' ), 
                         align = 'h', nrow=1, widths = c(4,1))
    # Annotate Figure
    combination1_ann = annotate_figure(
      combination1,
      right = legend
    )
    
    pdf(paste0('figures/03_figures/', figure_name,  mofa_configs$mofa_result[i],  '.pdf'), width =width_par, height =height_par)
    print(combination1_ann)
    dev.off()
    #print(combination1_ann)
    
    }
popup_function_pos(paste0('Generated:', ' figures/03_figures/ ', figure_name,  mofa_configs$mofa_result[i],  '.pdf'))  

In [75]:
## Save view colors for further usage

In [76]:
write.csv(data.frame(color_code = unlist(type_colors)),
          paste0('configurations/03_Type_Color_Codes.csv'))

In [77]:
## Inform that execution was finished
popup_function_pos('03_Run_MOFA: Execution Finished')

In [78]:
Sys.sleep(20)
popup_function_info('03_Run_MOFA')